# Faruq-v3 AF2RN — static + train-only observability — Kaggle

Attach the latest private dataset `faruq-v3-experiment-core-v1`, select a **T4 GPU**, enable **Internet**, then use **Run All**. The notebook requires the spectral-v2 SHA manifest, the opaque `.tar.bin` development archive, and the explicitly named `D0_seed42_best.pt`.

This notebook performs no training, reads no validation image/label/metric, and never accesses test. Input identity is checked before clone or package installation.

In [ ]:
import hashlib,json,os,time
from pathlib import Path
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
if not INPUT.is_dir() or not WORK.is_dir(): raise RuntimeError('Notebook ini khusus Kaggle.')
os.chdir(WORK)
print('INDEXING /kaggle/input ONCE ...',flush=True)
started=time.time(); index={}; files=0
for root,dirs,names in os.walk(INPUT):
    for name in names:
        path=Path(root)/name; index.setdefault(name,[]).append(path); files+=1
print(f'INPUT INDEX READY: {files} files in {time.time()-started:.2f}s',flush=True)
def one(name):
    matches=sorted(index.get(name,[]))
    if len(matches)!=1: raise FileNotFoundError(f'STOP CEPAT: harus tepat satu {name}; ditemukan {matches}. Refresh/attach versi terbaru faruq-v3-experiment-core-v1.')
    return matches[0]
MANIFEST=one('af2_spectral_kaggle_manifest.json')
ARCHIVE=one('faruq-development-v3-grouped.tar.bin')
D0_INPUT=one('D0_seed42_best.pt')
AF2_RESULT_INPUT=one('lfdet_afab_seed42_screening.json')
manifest=json.loads(MANIFEST.read_text(encoding='utf-8'))
if manifest.get('format')!='coffee_detector.af2_spectral.kaggle_manifest.v2': raise RuntimeError('STOP CEPAT: manifest bukan spectral-v2; update private Kaggle Dataset dahulu.')
if manifest.get('test_images_included') is not False: raise RuntimeError('STOP: bundle mengekspos test.')
for name,path in [('faruq-development-v3-grouped.tar.bin',ARCHIVE),('D0_seed42_best.pt',D0_INPUT),('lfdet_afab_seed42_screening.json',AF2_RESULT_INPUT)]:
    contract=manifest.get('artifacts',{}).get(name,{})
    if path.stat().st_size!=int(contract.get('bytes',-1)): raise RuntimeError(f'STOP CEPAT: ukuran {name} tidak cocok manifest.')
d0_proof=manifest.get('checkpoint_validation',{}).get('D0_seed42_best.pt',{})
if d0_proof.get('loadable_by_ultralytics') is not True or int(d0_proof.get('nc',-1))!=21: raise RuntimeError('STOP CEPAT: manifest tidak membuktikan D0 seed-42 sebagai checkpoint SNI-21 loadable.')
print('INPUT NAMES/SIZES PASS — belum clone/install/GPU work')
print('ARCHIVE:',ARCHIVE)
print('D0:',D0_INPUT)


In [ ]:
import importlib,importlib.metadata,shutil,subprocess,sys,torch
BRANCH='codex/af2-radially-normalized-angular-density'
REPO=WORK/'coffee-bean-detection'
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
for attempt in range(1,4):
    result=subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],cwd=WORK)
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==3: raise RuntimeError('git clone gagal tiga kali; cek Internet Kaggle.')
    time.sleep(2)
torch_before=importlib.metadata.version('torch')
subprocess.run([sys.executable,'-m','pip','install','-q','--disable-pip-version-check','ultralytics==8.4.96'],check=True,cwd=WORK)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True,cwd=WORK)
torch_after=importlib.metadata.version('torch')
if torch_after!=torch_before: raise RuntimeError(f'Instalasi mengubah Torch {torch_before} -> {torch_after}; restart session.')
for name in list(sys.modules):
    if name=='coffee_detector' or name.startswith('coffee_detector.'): sys.modules.pop(name,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan GPU Kaggle sebelum Run All.')
GPU=torch.cuda.get_device_name(0); CAPABILITY=torch.cuda.get_device_capability(0)
if CAPABILITY[0]<7: raise RuntimeError(f'GPU {GPU} tidak kompatibel; pilih T4 lalu restart session.')
try: probe=torch.ones(1,device='cuda:0').sum().item()
except Exception as exc: raise RuntimeError(f'Kernel CUDA gagal pada {GPU}: {exc}') from exc
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
print('BRANCH:',BRANCH); print('COMMIT:',COMMIT); print('GPU:',GPU); print('TORCH:',torch_after); print('ULTRALYTICS:',__import__('ultralytics').__version__)


In [ ]:
from coffee_detector.experiments.prepare_af2rn_kaggle import prepare_af2rn_kaggle_input
DATA,D0,CONTRACT=prepare_af2rn_kaggle_input(INPUT,WORK)
if CONTRACT.get('decision')!='PASS' or CONTRACT.get('validation_files_read') is not False or CONTRACT.get('test_images_accessed') is not False: raise RuntimeError('Kontrak input AF2RN gagal.')
if D0.resolve()!=D0_INPUT.resolve(): raise RuntimeError('Resolusi D0 berubah setelah instalasi.')
if (DATA/'test').exists(): raise RuntimeError('STOP: test tersedia.')
OUTPUT=WORK/'faruq-v3-af2rn-v1'; OUTPUT.mkdir(parents=True,exist_ok=True)
STATIC=OUTPUT/'static_audit.json'; OBS=OUTPUT/'observability_train.json'
(OUTPUT/'kaggle_input_contract.json').write_text(json.dumps(CONTRACT,indent=2)+'\n',encoding='utf-8')
print('FULL SHA/CONTRACT PASS')
print('DATA:',DATA); print('TRAIN:',CONTRACT['splits']['train']); print('VALIDATION FILES READ:',CONTRACT['validation_files_read']); print('D0 SHA256:',CONTRACT['d0_checkpoint_sha256']); print('OUTPUT:',OUTPUT)


In [ ]:
from coffee_detector.af2_rn.audit import run_af2rn_static_audit
static=run_af2rn_static_audit(D0,STATIC,device='cuda:0')
print('STATIC DECISION:',static['decision'])
print('PARAMETERS:',static['parameters'])
print('GATES:',static['gates'])
if static['decision']!='PASS': raise RuntimeError('STOP: static audit FAIL; observability/training dilarang.')


In [ ]:
from coffee_detector.af2_rn.observability import run_af2rn_observability_audit
observability=run_af2rn_observability_audit(DATA,DATA/'faruq_grouped_summary.json',STATIC,OBS,device='0',image_size=128,patches_per_image=16)
print('OBS DECISION:',observability['decision'])
print('IMAGES:',observability['images'])
print('NONDEGENERATE:',observability['nondegenerate_fraction'])
print('DIFFERS FROM AF2:',observability['different_from_af2_fraction'])
print('DISTRIBUTIONS:',observability['distributions'])
print('RADIAL:',observability['radial_retention'])
print('GATES:',observability['gates'])
print('TRAINING AUTHORIZED:',observability['training_authorized'])
if observability.get('validation_images_accessed') is not False or observability.get('test_images_accessed') is not False: raise RuntimeError('Audit melanggar data lock.')


In [ ]:
manifest_out={
    'format':'coffee_detector.af2rn.kaggle_audit_output.v1',
    'branch':BRANCH,'commit':COMMIT,'gpu':GPU,
    'd0_checkpoint_sha256':CONTRACT['d0_checkpoint_sha256'],
    'input_contract_decision':CONTRACT['decision'],
    'static_decision':static['decision'],
    'observability_decision':observability['decision'],
    'training_executed':False,'validation_files_read':False,'test_images_accessed':False,
}
(OUTPUT/'output_manifest.json').write_text(json.dumps(manifest_out,indent=2)+'\n',encoding='utf-8')
archive=shutil.make_archive(str(WORK/'faruq-v3-af2rn-audit-output'),'zip',root_dir=OUTPUT)
print('STATIC:',STATIC); print('OBSERVABILITY:',OBS); print('DOWNLOAD ZIP:',archive)
print('Kirim output static + observability. Jangan training; notebook ini tidak memiliki jalur training.')
